In [1]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import linen as nn
import flax.nnx as nnx
import optax
from tqdm import tqdm
import time 
from functools import partial
from jax import flatten_util
from VMC_tool import hi, edges,ha,SingleStateAnsatz,create_machine,compute_local_energies,\
    compute_qgt,forces_expect_hermitian,E_fcis

/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


H₂ FCI 基准能量
E0 = -1.01546825 Ha  |  激发能: 0.0000 eV
E1 = -0.87542794 Ha  |  激发能: 3.8107 eV
E2 = -0.42938376 Ha  |  激发能: 15.9482 eV
E3 = -0.26922131 Ha  |  激发能: 20.3064 eV


In [4]:
from pyscf import gto, scf, fci
import netket as nk
import netket.experimental as nkx

bond_length = 2.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()
print("="*60)
print("H₂ FCI 基准能量")
print("="*60)
for i, e in enumerate(E_fcis):
    exc = (e - E_fcis[0]) * 27.2114
    print(f"E{i} = {e:.8f} Ha  |  激发能: {exc:.4f} eV")

# ha = nkx.operator.from_pyscf_molecule(mol)
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1,1),
)
E_hf = mf.e_tot
print(f"HF 基态能量: {E_hf:.8f} Ha")

H₂ FCI 基准能量
E0 = -0.93725495 Ha  |  激发能: 0.0000 eV
E1 = -0.93098087 Ha  |  激发能: 0.1707 eV
E2 = -0.37473199 Ha  |  激发能: 15.3070 eV
E3 = -0.36644094 Ha  |  激发能: 15.5326 eV
HF 基态能量: -0.71591006 Ha


In [2]:
import jax
import jax.numpy as jnp
from functools import partial

# ==============================================
# 1. 生成随机初始态（你原有逻辑，保持不变）
# ==============================================
def generate_random_initial_states(hi, n_chains: int, seed: int = 42):
    key = jax.random.PRNGKey(seed)
    keys = jax.random.split(key, n_chains)
    return jax.vmap(lambda k: hi.random_state(k))(keys)

# ==============================================
# 2. 候选状态生成（固定 edges：(0,1),(2,3)）
# ==============================================
def make_get_all_next_states(edges):
    @jax.jit
    def get_all_next_states_jit(S: jnp.ndarray):
        next_states = []
        valid_masks = []
        for (i, j) in edges:
            occ_i = S[..., i]
            occ_j = S[..., j]
            valid = (occ_i != occ_j)
            new_state = S.at[..., i].set(occ_j).at[..., j].set(occ_i)
            next_states.append(new_state)
            valid_masks.append(valid)
        return jnp.stack(next_states), jnp.stack(valid_masks)
    return get_all_next_states_jit

# ==============================================
# 3. 单链 MH 步骤（纯函数，JIT 加速）
# ==============================================
def make_metropolis_hastings_step(edges, machine):
    get_all_next = make_get_all_next_states(edges)
    
    @jax.jit
    def mh_step(params, state: jnp.ndarray, key: jax.Array):
        candidates, valid_mask = get_all_next(state[None, :])
        candidates = candidates[:, 0]
        valid_mask = valid_mask[:, 0]
        
        # 随机选候选
        key, subk = jax.random.split(key)
        idx = jax.random.choice(subk, len(edges))
        cand = candidates[idx]
        is_valid = valid_mask[idx]
        
        # 接受率（复波函数正确公式）
        log_curr = machine(params, state)
        log_cand = machine(params, cand)
        log_acc = 2 * jnp.real(log_cand - log_curr)
        
        # 蒙特卡洛接受
        key, subk = jax.random.split(key)
        accept = is_valid & (log_acc > jnp.log(jax.random.uniform(subk)))
        
        # 更新状态
        new_state = jnp.where(accept, cand, state)
        return new_state, key
    
    return mh_step

@partial(jax.jit, static_argnums=(0,1,3,4,6,7))  # 新增 sweep_size 为静态参数
def mcmc_sampler_multichain(
    n_samples_per_chain: int,  # 每条链保存多少个样本
    n_warmup: int,             # 预烧步数（单位：sweep）
    initial_states: jnp.ndarray,
    edges: tuple,
    machine: callable,
    params: dict,
    seed: int = 42,
    sweep_size: int = 32       # 新增：每次样本保存前的跃迁次数
):
    n_chains = initial_states.shape[0]
    key = jax.random.PRNGKey(seed)
    chain_keys = jax.random.split(key, n_chains)
    
    mh_step = make_metropolis_hastings_step(edges, machine)
    
    # ---------------------
    # 单次 Sweep：执行 sweep_size 次跃迁
    # ---------------------
    @jax.jit
    def single_sweep(carry, _):
        states, keys = carry
        # 连续执行 sweep_size 次跃迁
        def body_fn(carry, _):
            s, k = carry
            new_s, new_k = jax.vmap(mh_step, in_axes=(None, 0, 0))(params, s, k)
            return (new_s, new_k), None
        
        (new_states, new_keys), _ = jax.lax.scan(
            body_fn, (states, keys), length=sweep_size
        )
        return (new_states, new_keys), new_states  # 只在 sweep 结束后保存样本
    
    # ---------------------
    # Warmup：执行 n_warmup 个 sweep，不保存样本
    # ---------------------
    (current_states, current_keys), _ = jax.lax.scan(
        single_sweep,
        (initial_states, chain_keys),
        length=n_warmup
    )
    
    # ---------------------
    # 采样：执行 n_samples_per_chain 个 sweep，每个 sweep 保存 1 个样本
    # ---------------------
    (_, _), samples = jax.lax.scan(
        single_sweep,
        (current_states, current_keys),
        length=n_samples_per_chain
    )
    
    # 形状：(n_samples_per_chain, n_chains, n_orbitals) → (总样本数, n_orbitals)
    return samples.reshape(-1, initial_states.shape[-1])

In [3]:
rngs = nnx.Rngs(21)
model = SingleStateAnsatz(4, hidden_dim=12, rngs=rngs)
machine, graphdef, params = create_machine(model)

samples = mcmc_sampler_multichain(
    n_samples_per_chain=200,
    n_warmup=100,
    initial_states=generate_random_initial_states(hi,16,2),
    edges=((0,1),(2,3)),
    machine=machine,
    params=params,
    seed=42,
    sweep_size=32
)
samples.shape


(3200, 4)

In [4]:
# ===================== 6. 初始化（适配多链） =====================
rngs = nnx.Rngs(21)
model = SingleStateAnsatz(4, hidden_dim=12, rngs=rngs)
machine, graphdef, params = create_machine(model)

optimizer = optax.sgd(learning_rate=0.01)
opt_state = optimizer.init(params)

# 训练参数（适配 Sweep 策略）
N_ITER = 300
N_CHAINS = 100  # 增加链数到和 NetKet 一致
N_SAMPLES_PER_CHAIN = 20  # 每条链保存 20 个样本 → 总样本数=100×20=2000
N_WARMUP = 10  # 预烧 10 个 sweep → 总预烧跃迁次数=10×32×100=32000
SWEEP_SIZE = 32  # 和 NetKet 一致

# ===================== 7. 训练循环（多链版本） =====================
print("\n" + "="*60)
print("开始多链 VMC 训练 (自然梯度下降法)")
print("="*60)

history = {
    'step': [],
    'energy': [],
    'energy_std': [],
    'error': []
}
start_time = time.time()
for step in range(N_ITER):
    # 1. 生成多链随机初始状态（模仿NetKet，无需手动指定单个initial_state）
    initial_states = generate_random_initial_states(hi, N_CHAINS, seed=21+step)  # 每次迭代换种子避免初始状态固定
    
    # 2. 多链采样（总样本数=16*63=1008，和原单链一致）
    samples = mcmc_sampler_multichain(
        n_samples_per_chain=N_SAMPLES_PER_CHAIN,
        n_warmup=N_WARMUP,
        initial_states=initial_states,
        edges=((0, 1), (2, 3)),
        machine=machine,
        params=params,
        seed=21+step,
        sweep_size=SWEEP_SIZE
    )
    
    # 3. 计算能量和自然梯度（逻辑和原代码一致）
    energy, energy_std, grad = forces_expect_hermitian(machine, params, samples)
    grad = jax.tree_map(lambda x: x*2, grad)
    qgt_reg,qgt_unravel_fun = compute_qgt(machine, params, samples, diag_shift=0.001) 
    grad_flat , grad_unravel_fn = flatten_util.ravel_pytree(grad)
  
    # 自然梯度求解
    natural_grad = jnp.linalg.solve(qgt_reg, grad_flat)
    natural_grad = grad_unravel_fn(natural_grad)
    grad = natural_grad
        
    # 4. 更新参数
    updates, opt_state = optimizer.update(grad, opt_state, params)
    params = optax.apply_updates(params, updates)
    
    # 5. 记录历史
    if step % 50 == 0 or step == N_ITER - 1:
        error = jnp.abs(energy.real - E_fcis[0])
        history['step'].append(step)
        history['energy'].append(float(energy.real))
        history['energy_std'].append(float(energy_std))
        history['error'].append(float(error))
        print(f"Step {step:3d} | E: {energy.real:.8f} ± {energy_std:.6f} | FCI: {E_fcis[0]:.8f} | Error: {error:.6f}")

end_time = time.time()
print(f"训练耗时：{end_time - start_time:.2f} 秒")
# 最终结果
final_energy, final_std, _ = forces_expect_hermitian(machine, params, samples)
final_error = jnp.abs(final_energy.real - E_fcis[0])
print("\n" + "="*60)
print(f"训练完成!")
print(f"最终能量：{final_energy.real:.8f} ± {final_std:.6f} Ha")
print(f"FCI 基准：{E_fcis[0]:.8f} Ha")
print(f"绝对误差：{final_error:.6f} Ha")
print(f"相对误差：{final_error / jnp.abs(E_fcis[0]) * 100:.4f}%")
print("="*60)


开始多链 VMC 训练 (自然梯度下降法)


/var/folders/8x/k_m4pmb11437ktb_r6tjzt2c0000gn/T/ipykernel_70923/2570571143.py:46: DeprecationWarning: jax.tree_map is deprecated: use jax.tree.map (jax v0.4.25 or newer) or jax.tree_util.tree_map (any JAX version).
  grad = jax.tree_map(lambda x: x*2, grad)


Step   0 | E: -0.48999252 ± 0.005455 | FCI: -1.01546825 | Error: 0.525476
Step  50 | E: -0.96665622 ± 0.002767 | FCI: -1.01546825 | Error: 0.048812
Step 100 | E: -1.00951567 ± 0.000813 | FCI: -1.01546825 | Error: 0.005953
Step 150 | E: -1.01393315 ± 0.000487 | FCI: -1.01546825 | Error: 0.001535
Step 200 | E: -1.01432403 ± 0.000361 | FCI: -1.01546825 | Error: 0.001144
Step 250 | E: -1.01456854 ± 0.000313 | FCI: -1.01546825 | Error: 0.000900
Step 299 | E: -1.01474501 ± 0.000273 | FCI: -1.01546825 | Error: 0.000723
训练耗时：164.77 秒

训练完成!
最终能量：-1.01491487 ± 0.000258 Ha
FCI 基准：-1.01546825 Ha
绝对误差：0.000553 Ha
相对误差：0.0545%
